In [1]:
!pip install flask werkzeug

In [2]:

from flask import Flask, render_template_string, request, redirect, url_for, flash
import sqlite3
import logging
from datetime import datetime, timedelta
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)
app.secret_key = "educational-demo-secret-change-before-deployment"

DATABASE = "password_security.db"

# Security logging
logging.basicConfig(
    filename="security.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

# Create database
def initialize_database():
    connection = sqlite3.connect(DATABASE)
    connection.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password_hash TEXT NOT NULL,
            failed_attempts INTEGER DEFAULT 0,
            lockout_until TEXT
        )
    """)
    connection.commit()
    connection.close()

initialize_database()

print("Flask application created successfully!")
print("Database initialized successfully!")


Flask application created successfully!
Database initialized successfully!


In [3]:

import re

def check_password_policy(password):
    if len(password) < 8:
        return False

    if not re.search(r"[A-Z]", password):
        return False

    if not re.search(r"[a-z]", password):
        return False

    if not re.search(r"[0-9]", password):
        return False

    if not re.search(r"[^A-Za-z0-9]", password):
        return False

    return True


def register_user(username, password):

    if not username.strip():
        return "Username cannot be empty."

    if not check_password_policy(password):
        return (
            "Password must contain at least 8 characters, "
            "uppercase, lowercase, number and special character."
        )

    password_hash = generate_password_hash(password)

    connection = sqlite3.connect(DATABASE)
    cursor = connection.cursor()

    try:
        cursor.execute(
            "INSERT INTO users (username, password_hash) VALUES (?, ?)",
            (username.strip(), password_hash)
        )

        connection.commit()

        logging.info("User registration successful: %s", username)

        return "Registration successful!"

    except sqlite3.IntegrityError:
        return "Username already exists."

    finally:
        connection.close()


print("Flask registration system created successfully!")


Flask registration system created successfully!


In [4]:
print(register_user("flaskstudent01", "Secure@123"))

Registration successful!


In [5]:

def login_user(username, password):

    connection = sqlite3.connect(DATABASE)
    cursor = connection.cursor()

    cursor.execute(
        """SELECT password_hash, failed_attempts, lockout_until
           FROM users WHERE username = ?""",
        (username,)
    )

    user = cursor.fetchone()

    if user is None:
        connection.close()
        logging.warning("Failed login for unknown username")
        return "Invalid username or password."

    password_hash, failed_attempts, lockout_until = user

    # Check account lockout
    if lockout_until:
        unlock_time = datetime.fromisoformat(lockout_until)

        if datetime.now() < unlock_time:
            connection.close()
            return "Account temporarily locked."

    # Verify password
    if check_password_hash(password_hash, password):

        cursor.execute(
            """UPDATE users
               SET failed_attempts = 0, lockout_until = NULL
               WHERE username = ?""",
            (username,)
        )

        connection.commit()
        connection.close()

        logging.info("Successful login: %s", username)

        return "Login successful!"

    else:
        failed_attempts += 1
        new_lockout = None

        if failed_attempts >= 5:
            new_lockout = (
                datetime.now() + timedelta(minutes=5)
            ).isoformat()

        cursor.execute(
            """UPDATE users
               SET failed_attempts = ?, lockout_until = ?
               WHERE username = ?""",
            (failed_attempts, new_lockout, username)
        )

        connection.commit()
        connection.close()

        logging.warning("Failed login: %s", username)

        if new_lockout:
            logging.warning("Account locked: %s", username)
            return "Account locked for 5 minutes."

        return "Invalid username or password."


print("Flask login system created successfully!")


Flask login system created successfully!


In [6]:
print(login_user("flaskstudent01", "Secure@123"))

Login successful!


In [7]:
print(login_user("flaskstudent01", "WrongPassword"))

Invalid username or password.


In [8]:

HTML_PAGE = """
<!DOCTYPE html>
<html>
<head>
    <title>Password Security System</title>

    <style>
        body {
            font-family: Arial;
            background: #eef2f7;
            padding: 30px;
        }

        .container {
            max-width: 600px;
            margin: auto;
            background: white;
            padding: 30px;
            border-radius: 12px;
        }

        h1, h2 {
            color: #245b91;
        }

        input {
            width: 95%;
            padding: 12px;
            margin: 10px 0;
        }

        button {
            background: #245b91;
            color: white;
            padding: 12px 20px;
            border: none;
            border-radius: 5px;
            cursor: pointer;
        }

        .message {
            color: #245b91;
            font-weight: bold;
        }
    </style>
</head>

<body>

<div class="container">

    <h1>Password Security System</h1>
    <p>Secure Authentication Demonstration</p>

    <hr>

    <h2>User Registration</h2>

    <form method="POST" action="/register">
        <input name="username"
               placeholder="Enter username"
               required>

        <input name="password"
               type="password"
               placeholder="Enter password"
               required>

        <button type="submit">Register</button>
    </form>

    <hr>

    <h2>User Login</h2>

    <form method="POST" action="/login">
        <input name="username"
               placeholder="Enter username"
               required>

        <input name="password"
               type="password"
               placeholder="Enter password"
               required>

        <button type="submit">Login</button>
    </form>

    <p class="message">{{ message }}</p>

    <hr>

    <h2>Security Features</h2>

    <ul>
        <li>Password Strength Checking</li>
        <li>Secure Password Hashing</li>
        <li>Failed Login Tracking</li>
        <li>Account Lockout</li>
        <li>Authentication Logging</li>
    </ul>

</div>

</body>
</html>
"""

print("Website HTML template created successfully!")


Website HTML template created successfully!


In [9]:

from flask import render_template_string, request

@app.route("/", methods=["GET"])
def home():
    return render_template_string(HTML_PAGE, message="")


@app.route("/register", methods=["POST"])
def register():
    username = request.form.get("username", "")
    password = request.form.get("password", "")

    message = register_user(username, password)

    return render_template_string(
        HTML_PAGE,
        message=message
    )


@app.route("/login", methods=["POST"])
def login():
    username = request.form.get("username", "")
    password = request.form.get("password", "")

    message = login_user(username, password)

    return render_template_string(
        HTML_PAGE,
        message=message
    )


print("Flask registration and login routes created!")


Flask registration and login routes created!


In [10]:

# Create a Flask test client
client = app.test_client()

# Test registration
registration_response = client.post(
    "/register",
    data={
        "username": "finaltest01",
        "password": "Secure@456"
    }
)

print("Registration test:")
print(registration_response.get_data(as_text=True).split(
    '<p class="message">')[1].split("</p>")[0])

# Test login
login_response = client.post(
    "/login",
    data={
        "username": "finaltest01",
        "password": "Secure@456"
    }
)

print("\nLogin test:")
print(login_response.get_data(as_text=True).split(
    '<p class="message">')[1].split("</p>")[0])


Registration test:
Registration successful!

Login test:
Login successful!


In [11]:

from google.colab.output import eval_js
from werkzeug.serving import make_server
import threading

# Start Flask server
server = make_server("0.0.0.0", 5000, app)

server_thread = threading.Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

# Generate a Colab access URL
url = eval_js("google.colab.kernel.proxyPort(5000)")

print("Password Security System is running!")
print("Open your website here:")
print(url)


Password Security System is running!
Open your website here:
https://5000-m-s-usc1b5-i41aei0ym2a-b.us-central1.prod.r.colab.dev
